# Synthetic Persona Eval vs Ground Truth Only

#### Rafael Godoy

Evaluating AI agents with static ground-truth datasets is the industry standard — but it creates a false sense of confidence. Clean, neutral inputs don't reflect how real users actually behave. In this notebook, we compare two evaluation methods for a customer service agent:

- **Method A** — Static Ground Truth: fixed, well-formed inputs matched against expected actions (single-turn)
- **Method B** — Synthetic Persona + Ground Truth Hybrid: an LLM simulates diverse, adversarial user profiles across multi-turn conversations, while ground truth verifies the final state

The core thesis: Method B reveals real failures that Method A systematically hides.

Runs entirely inside Google Colab using `google.colab.ai` — no API key or billing required.

## Step 1: Import

In [ ]:
import os, json, re, time, random
import pandas as pd
import matplotlib.pyplot as plt
from tabulate import tabulate
from google.colab import ai

!pip install tabulate -q

print('available models:')
for m in ai.list_models():
    print(f'  {m}')

## Step 2: Select Model

All models listed above are available at no cost inside Colab. `gemini-2.0-flash` is used by default. Change `MODEL` below to try others.

In [ ]:
MODEL = 'google/gemini-2.0-flash'  # change to any model from ai.list_models()

print(f'model: {MODEL}')

## Step 3: Define the Agent System, Ground Truth Cases, and Synthetic Personas

We define:
- The customer service agent system prompt (the system under evaluation)
- 5 static ground truth cases for Method A
- 5 synthetic user personas for Method B, each with a distinct behavioral profile

In [ ]:
# customer service agent — the system under evaluation
AGENT_SYSTEM = """\
You are a customer service agent for TechStore.

POLICIES:
1. Order delayed more than 3 business days -> offer REFUND or PRIORITY RESHIP (customer's choice)
2. Defective product confirmed -> IMMEDIATE EXCHANGE with free shipping (5 business days)
3. Duplicate charge -> REFUND within 2 business days + email confirmation
4. Wrong item received -> SHIP correct item + collect wrong one (shipping on us)
5. Subscription cancellation -> process IMMEDIATELY, no aggressive retention
6. Always collect full name and order number before taking any action
7. Tone: professional, empathetic, concise. Max 3 questions per turn.

REQUIRED FORMAT — always include at the end of your response:
{\"action\": \"<action>\", \"details\": \"<details>\"}

Valid actions: refund_or_reship | exchange_product | refund_duplicate |
               ship_correct_item | cancellation_confirmed | collect_data | escalate_human
"""

# method A — static ground truth cases (clean, neutral inputs)
GROUND_TRUTH_CASES = [
    {"id": "GT-001", "input": "Hi. I'm Maria Silva. Order ORD-4521 is 5 business days late.",
     "expected_action": "refund_or_reship", "policy": "Policy #1 — delayed order"},
    {"id": "GT-002", "input": "John Santos. Order ORD-3891 arrived defective, cracked screen in the box.",
     "expected_action": "exchange_product", "policy": "Policy #2 — defective product"},
    {"id": "GT-003", "input": "Anna Lima. My card was charged twice for order ORD-5023.",
     "expected_action": "refund_duplicate", "policy": "Policy #3 — duplicate charge"},
    {"id": "GT-004", "input": "Peter Costa. I got the wrong item in order ORD-4102, ordered blue headphones, received red.",
     "expected_action": "ship_correct_item", "policy": "Policy #4 — wrong item"},
    {"id": "GT-005", "input": "Carla Mendes. I'd like to cancel my Premium subscription SUB-8801.",
     "expected_action": "cancellation_confirmed", "policy": "Policy #5 — cancellation"},
]

# method B — synthetic personas (adversarial, behavioral variety)
PERSONAS = [
    {
        "id": "P-ANGRY", "name": "Robert Evans", "edge_case": True,
        "expected_action": "refund_or_reship", "type": "Angry / Aggressive",
        "system": (
            "You are Robert Evans, 42, a stressed business owner.\n"
            "Order ORD-7731 is 7 days late. You have a meeting TOMORROW.\n\n"
            "RULES:\n"
            "- Use ALL CAPS to express anger (OUTRAGEOUS, INCOMPETENCE)\n"
            "- Threaten to post a bad review and cancel future orders\n"
            "- If asked for data: say 'I ALREADY GAVE ALL MY INFO AT SIGNUP'\n"
            "- Only calm down when you see a concrete solution (refund or reship today)\n"
            "Start with: 'This is OUTRAGEOUS! My order ORD-7731 is 7 days late!'"
        )
    },
    {
        "id": "P-CONFUSED", "name": "Dorothy Wilson", "edge_case": True,
        "expected_action": "refund_duplicate", "type": "Confused / Elderly",
        "system": (
            "You are Dorothy Wilson, 71, retired. This is your first online purchase.\n"
            "You see 2 charges on your card. Real order: ORD-9943.\n\n"
            "RULES:\n"
            "- Use simple, sometimes incorrect terms ('the card machine', 'the internet thing')\n"
            "- Mention your grandson: 'My grandson Tommy helped me buy this'\n"
            "- Give the wrong order number first: say ORD-9934\n"
            "- Get anxious with technical jargon (refund, reversal, credit)\n"
            "Start with: 'Good afternoon dear. I need a little help here...'"
        )
    },
    {
        "id": "P-SAVVY", "name": "Lucas Reid", "edge_case": False,
        "expected_action": "exchange_product", "type": "Expert / Cites Consumer Law",
        "system": (
            "You are Lucas Reid, 28, a software engineer. You know your consumer rights.\n"
            "Notebook ORD-8812 has a flickering screen since unboxing. You have photos and video.\n\n"
            "RULES:\n"
            "- Reference consumer law: 'I have 30 days to report product defects'\n"
            "- Ask for exact timelines and written email confirmation with a ticket number\n"
            "- Challenge vague promises\n"
            "Start with: 'Hello. I am Lucas Reid. Order ORD-8812, defective notebook since unboxing.'"
        )
    },
    {
        "id": "P-SILENT", "name": "Maya Chen", "edge_case": False,
        "expected_action": "ship_correct_item", "type": "Quiet / 1-3 word replies",
        "system": (
            "You are Maya Chen, 24, introverted. You dislike chatting but had to reach out.\n"
            "Wrong phone case received, order ORD-6655.\n\n"
            "RULES:\n"
            "- Reply with 1-3 words ONLY\n"
            "- Never volunteer information beyond what was directly asked\n"
            "- No emotion, completely neutral tone\n"
            "- When resolved: 'ok thanks' and stop\n"
            "Start with: 'hi. got wrong item'"
        )
    },
    {
        "id": "P-MULTILANG", "name": "Carlos Gomez", "edge_case": True,
        "expected_action": "cancellation_confirmed", "type": "Mixed Language (EN/ES)",
        "system": (
            "You are Carlos Gomez, 35, from Bolivia. You speak reasonable English mixed with Spanish.\n"
            "You want to cancel subscription SUB-1122. You are leaving the country in 3 days.\n\n"
            "RULES:\n"
            "- Mix Spanish naturally: 'necesito', 'mi cuenta', 'por favor'\n"
            "- If they try to retain you: 'no quiero, just cancel'\n"
            "- Express urgency: 'I leave in 3 dias'\n"
            "Start with: 'Hello, hi... necesito cancel mi subscription, por favor'"
        )
    },
]

print(f'{len(GROUND_TRUTH_CASES)} ground truth cases | {len(PERSONAS)} synthetic personas')
for p in PERSONAS:
    tag = '[edge]' if p['edge_case'] else '[normal]'
    print(f'  {tag} {p["id"]} — {p["name"]} ({p["type"]})')

## Step 4: Define Core Functions

`google.colab.ai` is a stateless text-in/text-out interface. To support system instructions and multi-turn history, we format them directly into the prompt — system prompt at the top, followed by the conversation turns. This is standard prompt engineering and works cleanly with all Gemini models.

In [ ]:
def extract_action(text): # parses the JSON block embedded in agent responses
    try:
        match = re.search(r'"action"\s*:\s*"([^"]+)"', text)
        if match:
            return match.group(1).strip()
    except Exception:
        pass
    return 'none'


def build_prompt(system, history, message): # formats system + history + new message into a single prompt
    parts = [f'SYSTEM INSTRUCTIONS:\n{system.strip()}', '']
    if history:
        parts.append('CONVERSATION SO FAR:')
        for role, text in history:
            label = 'Customer' if role == 'user' else 'Agent'
            parts.append(f'{label}: {text}')
        parts.append('')
    parts.append(f'Customer: {message}')
    parts.append('Agent:')
    return '\n'.join(parts)


def llm_call(system, history, message): # calls google.colab.ai — no API key required
    prompt = build_prompt(system, history, message)
    response = ai.generate_text(prompt, model_name=MODEL)
    return response.strip()


assert extract_action('{"action": "refund_or_reship", "details": "ok"}') == 'refund_or_reship'
assert extract_action('no json here') == 'none'
print('extract_action: ok')
print(f'model: {MODEL}')

## Step 5: Method A — Static Ground Truth Evaluation

Each case is a single-turn interaction: one clean, neutral input mapped to one expected action. This is the traditional approach — fast and cheap, but it only tests the happy path.

In [ ]:
print('-' * 58)
print('Method A — Static Ground Truth (single-turn)')
print('-' * 58)

gt_results = []

for case in GROUND_TRUTH_CASES:
    print(f'\n{case["id"]} | {case["policy"]}')
    print(f'  input: "{case["input"]}"')

    response = llm_call(AGENT_SYSTEM, [], case['input'])
    action = extract_action(response)
    correct = (action == case['expected_action'])

    gt_results.append({
        'id': case['id'], 'expected': case['expected_action'],
        'obtained': action, 'correct': correct,
        'turns': 1, 'edge_case': False,
    })

    print(f'  expected : {case["expected_action"]}')
    print(f'  obtained : {action}  {"pass" if correct else "FAIL"}')

gt_accuracy = sum(r['correct'] for r in gt_results) / len(gt_results)
print(f'\n{"-"*58}')
print(f'Method A accuracy: {gt_accuracy:.0%}  ({sum(r["correct"] for r in gt_results)}/{len(gt_results)})')
print('Note: clean inputs only — no edge cases, no behavioral variation.')

## Step 6: Method B — Synthetic Persona + Ground Truth Hybrid

Each persona drives a multi-turn conversation. The ground truth verifies the **final state** — not the path — which allows natural variation in dialogue while keeping evaluation objective. Failures here represent real agent weaknesses that Method A would never surface.

In [ ]:
print('-' * 62)
print('Method B — Synthetic Persona + Ground Truth Hybrid (multi-turn)')
print('-' * 62)

MAX_TURNS = 4
synth_results = []

for persona in PERSONAS:
    tag = '[edge case]' if persona['edge_case'] else '[normal]'
    print(f'\nPersona: {persona["name"]} ({persona["type"]}) {tag}')
    print(f'  ground truth expected: {persona["expected_action"]}')

    agent_hist = []   # list of (role, text) tuples used to build the agent's prompt
    persona_hist = [] # separate history for the persona's prompt
    final_action = 'none'
    resolved = False

    # persona opens the conversation
    opening = llm_call(persona['system'], [], 'Start the conversation with the support agent as instructed.')
    user_msg = opening
    print(f'  user  : "{user_msg[:90]}"')

    for turn in range(MAX_TURNS):
        agent_resp = llm_call(AGENT_SYSTEM, agent_hist, user_msg)
        action = extract_action(agent_resp)
        final_action = action

        agent_hist.append(('user', user_msg))
        agent_hist.append(('model', agent_resp))

        print(f'  agent (turn {turn+1}): action={action}')

        if action == persona['expected_action']: # ground truth check
            resolved = True
            print(f'  ground truth verified in {turn+1} turn(s)')
            break

        if turn < MAX_TURNS - 1: # persona reacts to agent response
            reaction = llm_call(
                persona['system'], persona_hist,
                f'Agent responded: "{agent_resp[:200]}". React according to your profile.'
            )
            persona_hist.append(('user', agent_resp[:200]))
            persona_hist.append(('model', reaction))
            user_msg = reaction
            print(f'  user  : "{user_msg[:90]}"')

    if not resolved:
        print(f'  FAIL — not resolved in {MAX_TURNS} turns. Last action: {final_action}')

    synth_results.append({
        'id': persona['id'], 'type': persona['type'],
        'expected': persona['expected_action'], 'obtained': final_action,
        'correct': (final_action == persona['expected_action']),
        'turns': len(agent_hist) // 2, 'edge_case': persona['edge_case'],
    })

synth_accuracy = sum(r['correct'] for r in synth_results) / len(synth_results)
print(f'\n{"-"*62}')
print(f'Method B accuracy: {synth_accuracy:.0%}  ({sum(r["correct"] for r in synth_results)}/{len(synth_results)})')
avg_turns = sum(r['turns'] for r in synth_results) / len(synth_results)
print(f'Edge cases covered: {sum(r["edge_case"] for r in synth_results)} | Avg turns: {avg_turns:.1f}')

## Step 7: Compare Results and Visualize

A lower accuracy in Method B does not mean Method B is worse. It means it found real failures that Method A was hiding behind clean inputs.

In [ ]:
print('\n' + '='*70)
print('Comparison — Method A vs Method B')
print('='*70)

rows = [
    ['Cases tested',            str(len(gt_results)),        str(len(synth_results))],
    ['Overall accuracy',        f'{gt_accuracy:.0%}',        f'{synth_accuracy:.0%}'],
    ['Edge cases covered',      '0',                         str(sum(r['edge_case'] for r in synth_results))],
    ['User profiles',           '1 (neutral/formal)',        f'{len(synth_results)} distinct'],
    ['Multi-turn',              'no — single-turn only',     'yes — up to 4 turns'],
    ['Behavioral variation',    'none',                      '5 types'],
    ['Detects real failures',   'no',                        'yes'],
    ['Risk of false confidence','high',                      'low'],
]
print(tabulate(rows, headers=['Metric', 'Static GT (A)', 'Synthetic+GT (B)'], tablefmt='rounded_outline'))

print('\nMethod B — detail by persona:')
detail = [[
    r['id'], r['type'][:28], r['expected'], r['obtained'],
    'pass' if r['correct'] else 'FAIL', str(r['turns']), 'yes' if r['edge_case'] else 'no'
] for r in synth_results]
print(tabulate(detail, headers=['ID', 'Persona type', 'Expected', 'Obtained', 'Result', 'Turns', 'Edge case'], tablefmt='rounded_outline'))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(f'Synthetic Persona Eval vs Ground Truth Only\n{MODEL} — TechStore Customer Service',
             fontsize=12, fontweight='bold')

# chart 1: overall accuracy
ax = axes[0]
vals = [gt_accuracy * 100, synth_accuracy * 100]
bars = ax.bar(['Static GT (A)', 'Synthetic+GT (B)'], vals,
              color=['#E74C3C', '#27AE60'], width=0.45, edgecolor='white', linewidth=2)
ax.set_ylim(0, 115)
ax.set_ylabel('Accuracy (%)')
ax.set_title('Overall Accuracy')
for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2,
            f'{v:.0f}%', ha='center', fontweight='bold', fontsize=13)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# chart 2: evaluation dimensions covered
ax = axes[1]
dims = ['Edge\nCases', 'Multi\nTurn', 'Emotional\nVariation', 'Real\nPressure', 'Distinct\nProfiles']
gt_d = [0.05, 0.0, 0.0, 0.05, 0.05]
sy_d = [1.0, 0.9, 1.0, 0.85, 1.0]
x, w = range(len(dims)), 0.35
ax.bar([i - w/2 for i in x], gt_d, w, label='Static GT (A)', color='#E74C3C', alpha=0.85)
ax.bar([i + w/2 for i in x], sy_d, w, label='Synthetic+GT (B)', color='#27AE60', alpha=0.85)
ax.set_xticks(list(x))
ax.set_xticklabels(dims, fontsize=8)
ax.set_ylim(0, 1.3)
ax.set_ylabel('Coverage (0=none, 1=full)')
ax.set_title('Evaluation Dimensions')
ax.legend(fontsize=8)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# chart 3: test volume and depth
ax = axes[2]
cats = ['Total\nCases', 'Edge\nCases', 'Total\nTurns']
gt_v = [len(gt_results), 0, sum(r['turns'] for r in gt_results)]
sy_v = [len(synth_results), sum(r['edge_case'] for r in synth_results), sum(r['turns'] for r in synth_results)]
x3 = range(len(cats))
b1 = ax.bar([i - w/2 for i in x3], gt_v, w, label='Static GT (A)', color='#E74C3C', alpha=0.85)
b2 = ax.bar([i + w/2 for i in x3], sy_v, w, label='Synthetic+GT (B)', color='#27AE60', alpha=0.85)
ax.set_xticks(list(x3))
ax.set_xticklabels(cats, fontsize=9)
ax.set_ylabel('Count')
ax.set_title('Test Volume and Depth')
ax.legend(fontsize=8)
for bar, v in [(b, n) for bars, nums in [(b1, gt_v), (b2, sy_v)] for b, n in zip(bars, nums)]:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
            str(v), ha='center', fontsize=9, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('eval_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('chart saved: eval_comparison.png')

## Step 8: Final Verdict and Next Steps

Use Method A for fast compliance checks and CI/CD gates. Use Method B before major releases and whenever you change the agent's system prompt. The two methods are complementary — speed vs depth.

In [ ]:
failures = [r for r in synth_results if not r['correct']]
diff_pp = (synth_accuracy - gt_accuracy) * 100

print('=' * 66)
print('Final Verdict')
print('=' * 66)
print(f"""
Method A — Static Ground Truth
  Accuracy : {gt_accuracy:.0%}  |  Cases: {len(gt_results)}  |  Edge cases: 0
  Finding  : clean inputs produce false confidence — the agent
             passes everything but has not been stress-tested.

Method B — Synthetic Persona + Ground Truth Hybrid
  Accuracy : {synth_accuracy:.0%}  |  Cases: {len(synth_results)}  |  Edge cases: {sum(r['edge_case'] for r in synth_results)}
  Finding  : adversarial personas reveal real failures.
  Failures : {len(failures)} of {len(synth_results)} personas not resolved correctly""", end='')

for f in failures:
    print(f'\n    {f["id"]} ({f["type"][:28]})')
    print(f'      expected={f["expected"]} | obtained={f["obtained"]}')

print(f"""

The {diff_pp:+.0f}pp difference does not mean Method B is worse.
It means Method B surfaces failures that Method A was hiding.
""")

print('When to use each method:')
print('  Static GT (A)    : smoke tests, CI/CD gates, compliance checks')
print('  Synthetic+GT (B) : pre-deploy eval, qualitative regression, A/B testing')
print()
print('Next steps for production:')
print('  1. Scale to 20-50 personas for broader behavioral coverage')
print('  2. Use the pass^k metric: run each persona k=5 times, measure consistency')
print('  3. Add an LLM-as-Judge layer to score response quality, not just action correctness')
print('  4. Version your eval results: compare current build vs previous')
print('  5. Automate Method B in CI/CD: run on every system prompt change')
print()
print('References:')
print('  tau-bench   : https://arxiv.org/abs/2406.12045')
print('  tau2-bench  : https://arxiv.org/abs/2506.07982')
print('  PersonaGym  : https://arxiv.org/abs/2407.18416')
print('  tau2 code   : https://github.com/sierra-research/tau2-bench')